# TA: average scans and correct time zero

Example on 250709, scans `s40` and `s44`. It reproduces the lab's `s4044.dat` (check at the end).

Steps: read, inspect, average, subtract the pre-t0 spectrum, shift by the t0 curve, save.

Not shown: shot rejection and t0 picking, which are interactive (they will live in `pyspan`).
Here no shot is rejected, and the t0 curve is the one measured on `s20`-`s27`.

In [ ]:
import datetime
from pathlib import Path

import numpy as np

from edyx.calib import interp_calibration, read_calibration, write_calibration
from edyx.io import read_dat, stack_traces, write_dat
from edyx.plot import plot_maps
from edyx.tana import avg_scans, avg_spectra, shift_trace, subtract_pre_t0

## Parameters

In [ ]:
data_dir = Path("data")  # copied from 250709, see data/README.md
fnames = ["s40.dat", "s44.dat"]
t0_file = Path("output/250709_t0_s2027.toml")  # t0 curve of s20-s27, converted below
t_pre = -0.7  # ps: delays t < t_pre are before time zero

out_dir = Path("output")
out_root = "s4044"

## Legacy t0 file

The t0 curve of s20-s27 is stored in the old `_t0corr.npz` format. It is converted once to a
calibration file; new notebooks write one directly with `write_calibration`. The points were
picked on the measured wavelengths, hence `axis="wl_raw"`. The fit is kept for information.

In [ ]:
legacy = np.load(data_dir / "s2027_t0corr.npz")
t0_file.parent.mkdir(exist_ok=True)
write_calibration(
    t0_file, "t0", "wl_raw", *legacy["t0_corr"].T, units=["nm", "ps"],
    date=datetime.date(2025, 7, 9), source="s20-s27", made_by="s2027_t0corr.npz (converted)",
    fit={"model": "polynomial in eV, low order first", "coef": legacy["pol_coeff"],
         "points": legacy["t0_meas"]},
)

## Read

`stack_traces` checks that all files share the same delays and wavelengths. Axes of `dod`: (file, wl, delay).
NaN become masked points, so they are ignored by the averages.

Apply the wavelength calibration right after reading, so every later step uses the calibrated
axis: `wl = interp_calibration(read_calibration(wl_file), wl)`. Not here: the lab's output is
on the measured axis.

In [ ]:
t, wl, dod = stack_traces(read_dat(data_dir / fn) for fn in fnames)
dod = np.ma.masked_invalid(dod)
dod.shape

## Inspect the scans

After pre-t0 subtraction, for display only: the average below uses the raw scans.

In [ ]:
fig, axes = plot_maps(t, wl, subtract_pre_t0(t, dod, t_pre), titles=fnames)

## Average and subtract the pre-t0 spectrum

In [ ]:
avg = avg_scans(dod)
neg_spec = avg_spectra(avg[..., t < t_pre])  # mean spectrum before time zero, saved below
avg_sub = subtract_pre_t0(t, avg, t_pre)    # avg - neg_spec

fig, axes = plot_maps(t, wl, [avg, avg_sub], titles=["average", "pre-t0 subtracted"])

## Time-zero correction

The t0 curve is evaluated at the data wavelengths, by value. A wavelength outside the
range of the curve raises an error (`outside="error"`, the default).
`shift_trace` moves the signal at `t0[i]` to delay 0, and keeps only the delays valid for every wavelength.

In [ ]:
t0 = interp_calibration(read_calibration(t0_file), wl)

t_c, shifted = shift_trace(t, avg_sub, t0)

In [ ]:
fig, axes = plot_maps(t, wl, [avg_sub], titles=["before"])
axes[0, 0].plot(t0, wl, color="k", lw=0.5)
axes[0, 0].set_xlim(-2, 2)

fig, axes = plot_maps(t_c, wl, [shifted], titles=["after"])
axes[0, 0].axvline(0, color="k", lw=0.5)
axes[0, 0].set_xlim(-2, 2)

## Save

In [ ]:
out_dir.mkdir(exist_ok=True)
write_dat(out_dir / f"{out_root}.dat", t_c, wl, shifted,
          header=f"scans: {fnames}\nt_pre: {t_pre} ps\nt0 curve: {t0_file.name}")
np.savetxt(out_dir / f"{out_root}_negdelay.dat", np.column_stack([wl, neg_spec.filled(np.nan)]))

fig, axes = plot_maps(t_c, wl, [shifted], vmin=-15, vmax=15)
fig.savefig(out_dir / f"{out_root}.png", dpi=300)

## Check against the lab's output

In [ ]:
t_ref, wl_ref, ref = read_dat(data_dir / f"{out_root}.dat")
neg_ref = np.loadtxt(data_dir / f"{out_root}_negdelay.dat")[:, 1]
assert np.allclose(t_c, t_ref) and np.allclose(wl, wl_ref)
print("max |difference|:", np.max(np.abs(shifted - ref)))  # files hold 7 significant digits
assert np.ma.allclose(shifted, ref, rtol=1e-6, atol=1e-4)
assert np.ma.allclose(neg_spec, neg_ref)